In [ ]:
import os, json
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

import sys
sys.path.insert(0, '.')
from utils import load_sample, RANDOM_STATE, TEST_SIZE, RESULTS_DIR

os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
df = load_sample()
print(df.shape)
print(df['label'].value_counts())

In [ ]:
vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2, 4), max_features=20_000, sublinear_tf=True)
X = vectorizer.fit_transform(df['domain'])
y = df['label'].values
print(X.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_features='sqrt', n_jobs=-1, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=['benign', 'dga']))
print(f'ROC-AUC : {roc_auc_score(y_test, y_prob):.4f}')

In [ ]:
joblib.dump(rf, os.path.join(RESULTS_DIR, 'rf_default.joblib'))
joblib.dump(vectorizer, os.path.join(RESULTS_DIR, 'tfidf_vectorizer.joblib'))
print('Saved.')